In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import os
import re
import xarray as xr
from read_CT import*
from socket import gethostname
import gsw

%load_ext autoreload
%autoreload 2
%matplotlib inline

os.chdir("../../..")
savedir = os.path.join(os.getcwd(),"DATA")
path = os.getcwd()
path_adcp = os.path.join(path,"DATA/SIOS21/adcp_data_and_analysis_lars_smedsrud/")
ds_adcp = xr.open_dataset(path_adcp+"Nortek_ADCP_currents_Oct20_to_Nov4_avgd_cal.nc", engine="netcdf4")

In [3]:
path_CT = os.path.join(path,"DATA/SIOS21/Ant2021/Ant2021CT/", "CT_data_small.nc")
if os.path.exists(path_CT):
    ds = xr.open_dataset(path_CT, engine='netcdf4')
else:
    print("reading CNV files")
    ds = read_write_all_cnv(os.path.join(path_CT))
# downsample to ADCP temporal resolution for easier checking signs
ds = ds.resample(time="10min").mean()

In [8]:
lat = -77.8667
lon = 166.23333333333332


# ---- shorthand ----
ds = ds.copy()   # your dataset

# ---- pressure: TEOS-10 expects positive dbar ----
p = np.abs(ds.Pressure_db)

# ---- conductivity: S/m -> mS/cm ----
C_mScm = 10.0 * ds.Conductivity_S_m

# ---- Practical Salinity (PSS-78) ----
SP = gsw.SP_from_C(
    C=C_mScm,
    t=ds.Temperature_DegC,  # ITS-90 in-situ temperature
    p=p
)

# ---- Absolute Salinity (TEOS-10) ----
SA = gsw.SA_from_SP(
    SP=SP,
    p=p,
    lon=lon,
    lat=lat
)

# ---- Conservative Temperature (TEOS-10) ----
CT = gsw.CT_from_t(
    SA=SA,
    t=ds.Temperature_DegC,  # ITS-90
    p=p
)

# ---- Freezing-point Conservative Temperature ----
CT_freeze = gsw.CT_freezing(
    SA=SA,
    p=p,
    saturation_fraction=0.0
)

# ---- In-situ supercooling (positive => supercooled) ----
supercooling = CT_freeze - CT

In [10]:
ds["SP"] = SP
ds["SA"] = SA
ds["CT"] = CT
ds["CT_freezing"] = CT_freeze
ds["supercooling"] = supercooling

In [14]:
if os.path.exists(os.path.join(path,"DATA/Ant2021/Ant2021CT/CT_data_small_teos10.nc")):
    os.remove(os.path.join(path,"DATA/Ant2021/Ant2021CT/CT_data_small_teos10.nc"))
ds.to_netcdf(os.path.join(path,"DATA/Ant2021/Ant2021CT/CT_data_small_teos10.nc"), engine="netcdf4")